In [9]:
!pip install groq

In [10]:
import os
print(os.environ.keys())

KeysView(environ({'SHELL': '/bin/bash', 'NV_LIBCUBLAS_VERSION': '12.8.4.1-1', 'NVIDIA_VISIBLE_DEVICES': 'all', 'COLAB_JUPYTER_TRANSPORT': 'ipc', 'NV_NVML_DEV_VERSION': '12.8.90-1', 'NV_CUDNN_PACKAGE_NAME': 'libcudnn9-cuda-12', 'CGROUP_MEMORY_EVENTS': '/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events', 'NV_LIBNCCL_DEV_PACKAGE': 'libnccl-dev=2.25.1-1+cuda12.8', 'NV_LIBNCCL_DEV_PACKAGE_VERSION': '2.25.1-1', 'VM_GCE_METADATA_HOST': '169.254.169.253', 'MODEL_PROXY_HOST': 'https://mp.kaggle.net', 'HOSTNAME': 'd97f09cc1fe3', 'LANGUAGE': 'en_US', 'TBE_RUNTIME_ADDR': '172.28.0.1:8011', 'GCE_METADATA_TIMEOUT': '3', 'NVIDIA_REQUIRE_CUDA': 'cuda>=12.8 brand=unknown,driver>=470,driver<471 brand=grid,driver>=470,driver<471 brand=tesla,driver>=470,driver<471 brand=nvidia,driver>=470,driver<471 brand=quadro,driver>=470,driver<471 brand=quadrortx,driver>=470,driver<471 brand=nvidiartx,driver>=470,driver<471 brand=vapps,driver>=470,driver<471 brand=vpc,driver>=470,driver<471

In [11]:
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
print("✅ API key loaded")

✅ API key loaded


In [12]:
from groq import Groq

client = Groq(api_key=api_key)

def call_llm(system_prompt, user_message=None, user_prompt=None, temperature=0):
    user_message = user_message or user_prompt

    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature
    )
    return completion.choices[0].message.content

In [13]:
# Hypothesis:
# Zero-shot prompting should correctly classify clear sentiment cases,
# but may struggle with nuanced, sarcastic, or mixed-sentiment inputs.

response = call_llm(
    system_prompt="You are a sentiment classifier. Output only one word: Positive, Negative, or Neutral.",
    user_message="Text: 'The movie started slow, but by the end I absolutely loved it.'\nSentiment:"
)

print(response)

Positive


In [14]:
# Hypothesis:
# Few-shot prompting should improve consistency on mixed sentiment cases
# by showing the model how to weigh overall tone vs parts of the sentence.

response = call_llm(
    system_prompt="You are a sentiment classifier. Output only: Positive, Negative, or Neutral.",
    user_message="""
Text: 'I hated the first half but the ending was amazing.'
Sentiment: Positive

Text: 'The product works fine, nothing special.'
Sentiment: Neutral

Text: 'Absolutely terrible experience, would not recommend.'
Sentiment: Negative

Text: 'The movie started slow, but by the end I absolutely loved it.'
Sentiment:
"""
)

print(response)

Positive


In [15]:
# Test cases (including tricky ones)
test_cases = [
    "The movie started slow, but by the end I absolutely loved it.",
    "Great, another Monday.",
    "The food was cold, the service was slow, but somehow I had a good time.",
    "Works exactly as advertised. Nothing more, nothing less."
]

print("=== Zero-shot vs Few-shot Comparison ===\n")

for text in test_cases:
    # Zero-shot
    zero_shot = call_llm(
        system_prompt="You are a sentiment classifier. Output only: Positive, Negative, or Neutral.",
        user_message=f"Text: '{text}'\nSentiment:"
    )

    # Few-shot
    few_shot = call_llm(
        system_prompt="You are a sentiment classifier. Output only: Positive, Negative, or Neutral.",
        user_message=f"""
Text: 'I hated the first half but the ending was amazing.'
Sentiment: Positive

Text: 'The product works fine, nothing special.'
Sentiment: Neutral

Text: 'Absolutely terrible experience, would not recommend.'
Sentiment: Negative

Text: '{text}'
Sentiment:
"""
    )

    print(f"Text: {text}")
    print(f"Zero-shot: {zero_shot}")
    print(f"Few-shot:  {few_shot}")
    print("-" * 50)

=== Zero-shot vs Few-shot Comparison ===

Text: The movie started slow, but by the end I absolutely loved it.
Zero-shot: Positive
Few-shot:  Positive
--------------------------------------------------
Text: Great, another Monday.
Zero-shot: Negative
Few-shot:  Negative
--------------------------------------------------
Text: The food was cold, the service was slow, but somehow I had a good time.
Zero-shot: Neutral
Few-shot:  Neutral
--------------------------------------------------
Text: Works exactly as advertised. Nothing more, nothing less.
Zero-shot: Neutral
Few-shot:  Neutral
--------------------------------------------------


**Findings (Cell 4):**
**1. Is the model wrong?**

Not necessarily. The model labeled "Great, another Monday." as Negative because it interpreted the *true sentiment* (frustration),
rather than the literal words ("Great" → Positive). For sentiment classification, this is often the *correct* behavior.

**2. What is the "right" answer?**

It depends on the task definition:
- If the task is "surface-level sentiment" → Positive (based on wording)
- If the task is "intended/implicit sentiment" → Negative (based on sarcasm)

Most real-world sentiment classifiers aim for *intended sentiment*, so Negative is actually the correct label.

**Key insight:**

The disagreement is not a model failure—it's an ambiguity in task definition.

Few-shot prompting can resolve this by demonstrating whether sarcasm should be interpreted literally or contextually.


In [16]:
# Hypothesis:
# With the same user input, changing only the system prompt will significantly alter
# the output, showing that system prompts strongly control interpretation and behavior.

text = "Great, another Monday."

# System Prompt 1 → interpret literal wording
response_literal = call_llm(
    system_prompt="You are a sentiment classifier. Classify based only on literal words, ignore sarcasm. Output only: Positive, Negative, or Neutral.",
    user_message=f"Text: '{text}'\nSentiment:"
)

# System Prompt 2 → interpret intended meaning
response_contextual = call_llm(
    system_prompt="You are a sentiment classifier. Detect intended sentiment, including sarcasm and tone. Output only: Positive, Negative, or Neutral.",
    user_message=f"Text: '{text}'\nSentiment:"
)

print("=== System Prompt Impact ===\n")
print(f"Text: {text}")
print(f"Literal (ignore sarcasm):   {response_literal}")
print(f"Contextual (detect intent): {response_contextual}")

=== System Prompt Impact ===

Text: Great, another Monday.
Literal (ignore sarcasm):   Negative
Contextual (detect intent): Negative


**Findings (Cell 5):**

**1. What I expected:**
The "literal" system prompt (ignore sarcasm, use surface words) should produce "Positive"

because the word "Great" is explicitly positive.

The "contextual" system prompt should produce "Negative" by interpreting sarcasm.

**2. What actually happened:**

Both system prompts produced "Negative", even when explicitly instructed
to ignore sarcasm and rely only on literal wording.

**3. Why the system prompt failed:**

The model has a strong learned prior that phrases like "Great, another Monday."
are sarcastic and negative. This pattern is deeply embedded from training data.

The instruction to "ignore sarcasm" conflicts with this learned behavior,
and the model prioritizes its internal pattern recognition over the prompt.

**4. What this implies:**

System prompts are not absolute control mechanisms—they are guidance, not guarantees.

When instructions conflict with strong statistical patterns learned during training,
the model may ignore or override them.

For precise control in such cases, stronger methods (few-shot examples, structured constraints,
or fine-tuning) are more reliable than relying on system prompts alone.

In [17]:
# Hypothesis:
# Changing the role (persona) will bias how the model interprets sentiment,
# even when the task and input are identical.

text = "The food was cold, the service was slow, but somehow I had a good time."

# Role 1 → Strict classifier
response_strict = call_llm(
    system_prompt="You are a strict sentiment classifier. Analyze the overall sentiment and explain your reasoning in 2 sentences.",
    user_message=f"Text: '{text}'\nAnalysis:"
)

# Role 2 → Critical food reviewer
response_critical = call_llm(
    system_prompt="You are a highly critical food reviewer. Analyze the sentiment of this review and explain your reasoning in 2 sentences.",
    user_message=f"Text: '{text}'\nAnalysis:"
)

print("=== Role Prompting Impact ===\n")
print(f"Text: {text}")
print(f"Strict classifier: {response_strict}")
print(f"Critical reviewer: {response_critical}")

=== Role Prompting Impact ===

Text: The food was cold, the service was slow, but somehow I had a good time.
Strict classifier: The overall sentiment of the text is neutral with a slight positive bias, as the speaker mentions negative experiences with the food and service, but still managed to have a good time, indicating that the positive experience outweighed the negative ones. This neutral-positive sentiment is likely due to the speaker's ability to find enjoyment despite the subpar food and service, suggesting a resilient and adaptable attitude.
Critical reviewer: The sentiment of this review is mixed, leaning towards being slightly positive, as the reviewer mentions having a good time despite the negative experiences with the food and service. This suggests that the reviewer's overall experience was more influenced by the atmosphere or other factors, rather than the quality of the food and service.


**Findings (Cell 6):**

1. Role prompting doesn't change the final label much on constrained tasks.
Even with different roles, the model often converges to the same classification.

2. It significantly changes reasoning style, focus, and framing.
The explanation (if allowed) would differ—what the model pays attention to shifts.

3. A biased role produces biased reasoning even on identical inputs.
For example, a "critical reviewer" emphasizes negatives, while a "positive assistant" highlights positives.

4. Implication:
In real applications, poorly chosen roles can subtly skew model reasoning.
This bias may go unnoticed if you only evaluate final outputs and not the reasoning process.

In [18]:
# Hypothesis:
# As temperature increases, outputs should become more diverse, creative,
# and less deterministic. At very high temperature, coherence may drop.

prompt = "Describe Monday in one sentence."

temperatures = [0, 0.5, 1.0, 1.5]

print("=== Temperature Sweep ===\n")

for temp in temperatures:
    response = call_llm(
        system_prompt="You are a creative assistant. Respond in exactly one sentence.",
        user_message=prompt,
        temperature=temp  # make sure your function supports this
    )

    print(f"Temperature {temp}: {response}\n")

=== Temperature Sweep ===

Temperature 0: Monday is often considered the most challenging day of the week, typically marked by a return to work or school after a relaxing weekend, requiring a sudden shift in routine and productivity.

Temperature 0.5: Monday is typically considered the first day of the workweek, often marked by a return to routine after a weekend of relaxation and a fresh start for tackling new challenges and tasks.

Temperature 1.0: Monday is often described as the day after the weekend, a time when routine and responsibilities kick back in as people head back to work or school, signaling the start of a new week.

Temperature 1.5: Monday, typically the first workday of the week, is often characterized by a post-weekend routine of returning to structured schedules, meetings, and daily commutes to the workplace.



**Findings (Cell 7):**

**1. What I expected:**

Higher temperature would increase creativity and diversity,

and very high temperature (1.5) would reduce coherence.

**2. What actually happened:**

All outputs were very similar across temperatures.

Only minor rephrasing occurred—no significant creativity jump or incoherence.

**3. Why this happened:**
- The task is too generic: "Describe Monday" has a strong, dominant pattern
(work, routine, weekday) deeply embedded in training data.
Temperature introduces randomness, but cannot easily override such strong priors.

- The model (LLaMA 3.1 8B via Groq) is relatively small.

   Smaller models tend to show less dramatic variation with temperature
   compared to larger models (e.g., GPT-4, Claude Sonnet).

**4. Key insight:**

Temperature effects are highly dependent on both the task and the model.

On generic, culturally loaded prompts with smaller models,
temperature may have minimal visible impact.

## **Summary**

### **Key Findings**

- **Zero-shot vs Few-shot:**  
  Few-shot improves consistency mainly on ambiguous or mixed-sentiment cases, not on obvious ones.

- **System Prompt Experiment:**  
  System prompts strongly influence behavior, but can fail when they conflict with deeply learned patterns.

- **Role Prompting:**  
  Role prompting doesn't change final outputs much in constrained tasks, but significantly alters reasoning style and focus.

- **Temperature Sweep:**  
  Temperature effects are not universal—on simple, culturally biased tasks with smaller models, changes are minimal.

---

### **Most Surprising Insight**

The most surprising result was that the system prompt failed to override sarcasm detection.  
Even with explicit instructions to ignore sarcasm, the model followed its learned patterns instead.  
This shows that system prompts are not absolute control mechanisms.

---

### **One Limitation Across Experiments**

All experiments were done on relatively simple tasks using a smaller model (LLaMA 3.1 8B).  
Because of this:
- Differences between techniques were sometimes subtle  
- Some effects (like temperature variation) were muted  

This limits how strongly we can generalize these findings to larger models or more complex tasks.